In [1]:
import pandas as pd
import geopandas as gpd
import requests as re
from io import BytesIO
from os import path, makedirs

from core.downloads.orcamento import load_orcamento, load_orcamento_r

## Orçamento previsto/liquidado da função habitação, programa 3008 e projetos atividades 1701 e 1702

Vamos começar coletando os dados orçamentários do site de execução orçamentária da Secretaria da Fazenda.

In [2]:
ANOS = [2024, 2025]

df_orcamento = pd.concat(
    [load_orcamento(ano).assign(ANO=str(ano)) for ano in ANOS],
    ignore_index=True
)
df_orcamento

,DataInicial,DataFinal,Cd_AnoExecucao,Cd_Exercicio,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_Descongelado,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao,ANO
0,01/01/2024,31/12/2024,2024,2024,169108,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,1000,0,0,0,0,1000,18/01/2025,2024
1,01/01/2024,31/12/2024,2024,2024,173723,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,"1317978,6",1314177,1314177,1314177,1314177,"3801,6",18/01/2025,2024
2,01/01/2024,31/12/2024,2024,2024,171046,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,"3131324,4","3131324,4","3131324,4","2940879,9","2791266,85",0,18/01/2025,2024
3,01/01/2024,31/12/2024,2024,2024,173862,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,"37258939,15","37222552,12","37222552,12","30999672,71","30291725,36","36387,03",18/01/2025,2024
4,01/01/2024,31/12/2024,2024,2024,180471,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,2000000,2000000,2000000,2000000,2000000,0,18/01/2025,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17409,01/01/2025,31/12/2025,2025,2025,181532,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,0,660000,0,0,0,0,660000,16/01/2026,2025
17410,01/01/2025,31/12/2025,2025,2025,181534,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,0,635000,"100819,63","100819,63","100220,06","100220,06","534180,37",16/01/2026,2025
17411,01/01/2025,31/12/2025,2025,2025,181535,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,0,132000,0,0,0,0,132000,16/01/2026,2025
17412,01/01/2025,31/12/2025,2025,2025,181543,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,0,215000,0,0,0,0,215000,16/01/2026,2025


In [3]:
for col in [col for col in df_orcamento.columns if 'Vl' in col]:
    df_orcamento.loc[:, col] = df_orcamento[col].str.replace(',', '.').astype(float)
df_orcamento.loc[:, 'DataExtracao'] = pd.to_datetime(df_orcamento['DataExtracao'], format='%d/%m/%Y')
df_orcamento

,DataInicial,DataFinal,Cd_AnoExecucao,Cd_Exercicio,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_Descongelado,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao,ANO
0,01/01/2024,31/12/2024,2024,2024,169108,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18 00:00:00,2024
1,01/01/2024,31/12/2024,2024,2024,173723,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,"1317978,6",1314177.0,1314177.0,1314177.0,1314177.0,"3801,6",2025-01-18 00:00:00,2024
2,01/01/2024,31/12/2024,2024,2024,171046,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,"3131324,4",3131324.4,3131324.4,2940879.9,2791266.85,0,2025-01-18 00:00:00,2024
3,01/01/2024,31/12/2024,2024,2024,173862,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,"37258939,15",37222552.12,37222552.12,30999672.71,30291725.36,"36387,03",2025-01-18 00:00:00,2024
4,01/01/2024,31/12/2024,2024,2024,180471,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,2000000,2000000.0,2000000.0,2000000.0,2000000.0,0,2025-01-18 00:00:00,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17409,01/01/2025,31/12/2025,2025,2025,181532,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0.0,0.0,660000,0.0,0.0,0.0,0.0,660000,2026-01-16 00:00:00,2025
17410,01/01/2025,31/12/2025,2025,2025,181534,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0.0,0.0,635000,100819.63,100819.63,100220.06,100220.06,"534180,37",2026-01-16 00:00:00,2025
17411,01/01/2025,31/12/2025,2025,2025,181535,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0.0,0.0,132000,0.0,0.0,0.0,0.0,132000,2026-01-16 00:00:00,2025
17412,01/01/2025,31/12/2025,2025,2025,181543,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0.0,0.0,215000,0.0,0.0,0.0,0.0,215000,2026-01-16 00:00:00,2025


In [4]:
filtro_orcamento = (
    (df_orcamento['Cd_Funcao']=='16') |
    (df_orcamento['Cd_Programa']=='3008') |
    (df_orcamento['ProjetoAtividade'].isin(['1702', '1703']))
)

df_orcamento = df_orcamento.loc[filtro_orcamento]
df_orcamento

,DataInicial,DataFinal,Cd_AnoExecucao,Cd_Exercicio,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_Descongelado,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao,ANO
27,01/01/2024,31/12/2024,2024,2024,169113,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18 00:00:00,2024
28,01/01/2024,31/12/2024,2024,2024,167099,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18 00:00:00,2024
29,01/01/2024,31/12/2024,2024,2024,167101,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18 00:00:00,2024
30,01/01/2024,31/12/2024,2024,2024,171051,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,14985177,14963773.14,14963773.14,14842626.58,14828557.72,"21403,86",2025-01-18 00:00:00,2024
31,01/01/2024,31/12/2024,2024,2024,173960,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,10000000,9968965.45,9968965.45,6827361.22,6827361.22,"31034,55",2025-01-18 00:00:00,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17275,01/01/2025,31/12/2025,2025,2025,183812,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.0,0.0,"7599391,97",7599391.86,7599391.86,6795448.86,6795448.86,"0,11",2026-01-16 00:00:00,2025
17276,01/01/2025,31/12/2025,2025,2025,175603,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.0,241597.0,1369054,0.0,0.0,0.0,0.0,1369054,2026-01-16 00:00:00,2025
17277,01/01/2025,31/12/2025,2025,2025,181553,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,973387.53,0.0,"7564444,52",7470213.51,7470213.51,4815239.24,4815239.24,"94231,01",2026-01-16 00:00:00,2025
17278,01/01/2025,31/12/2025,2025,2025,175604,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.0,2938459.0,"10014806,08",2976788.58,2976788.58,2588105.58,2588105.58,"7038017,5",2026-01-16 00:00:00,2025


## Orçamento regionalizado no Programa 3002

O orçamento é regionalizado apenas na liquidação, então não é possível obter o orçamento previsto por subprefeitura, mas é possível obter o orçamento liquidado.

In [5]:
df_orcamento_r = pd.concat(
    [load_orcamento_r(ano).assign(ANO=str(ano)) for ano in ANOS],
    ignore_index=True
)
df_orcamento_r

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,ANO
0,01,19738,2024,1799,2025,2025-01-13 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625,2024
1,01,19738,2024,114583,2024,2024-04-26 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625,2024
2,01,19738,2024,124915,2024,2024-05-09 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625,2024
3,01,19738,2024,157086,2024,2024-06-14 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625,2024
4,01,19738,2024,187349,2024,2024-07-17 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1114314,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,"10,11",2025
1114315,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Norte,Subprefeitura Santana/Tucuruvi,Supra-Distrital,Despesa Regionalizável,"55,51",2025
1114316,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Norte,Subprefeitura Vila Maria/Vila Guilherme,Supra-Distrital,Despesa Regionalizável,"14,17",2025
1114317,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,"28,33",2025


In [6]:
df_orcamento_r.loc[:, 'VALOR_DETALHAMENTO_AÇÃO'] = df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'].str.replace(',', '.').astype(float)
df_orcamento_r

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,ANO
0,01,19738,2024,1799,2025,2025-01-13 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625.0,2024
1,01,19738,2024,114583,2024,2024-04-26 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625.0,2024
2,01,19738,2024,124915,2024,2024-05-09 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625.0,2024
3,01,19738,2024,157086,2024,2024-06-14 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625.0,2024
4,01,19738,2024,187349,2024,2024-07-17 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,8625.0,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1114314,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,10.11,2025
1114315,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Norte,Subprefeitura Santana/Tucuruvi,Supra-Distrital,Despesa Regionalizável,55.51,2025
1114316,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Norte,Subprefeitura Vila Maria/Vila Guilherme,Supra-Distrital,Despesa Regionalizável,14.17,2025
1114317,01,9335,2025,335055,2025,12/19/2025 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,28.33,2025


In [7]:
filtro_orcamento_r = (
    (df_orcamento_r['CÓDIGO_FUNÇÃO']=='16') |
    (df_orcamento_r['CÓDIGO_PROGRAMA']=='3008') |
    (df_orcamento_r['CÓDIGO_PROJ_ATIV'].isin(['1702', '1703']))
)

df_orcamento_r = df_orcamento_r.loc[filtro_orcamento_r]
df_orcamento_r = df_orcamento_r.loc[df_orcamento_r['ANO_LIQUIDAÇÃO']==df_orcamento_r['ANO_EMPENHO']]
df_orcamento_r

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,ANO
47,01,19741,2024,67338,2024,2024-03-11 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,713.77,2024
48,01,19741,2024,67338,2024,2024-03-11 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Leste,Subprefeitura Aricanduva/Formosa/Carrão,Supra-Distrital,Despesa Regionalizável,89.35,2024
49,01,19741,2024,67338,2024,2024-03-11 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Leste,Subprefeitura Cidade Tiradentes,Supra-Distrital,Despesa Regionalizável,86.22,2024
50,01,19741,2024,67338,2024,2024-03-11 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Leste,Subprefeitura Ermelino Matarazzo,Supra-Distrital,Despesa Regionalizável,85.93,2024
51,01,19741,2024,67338,2024,2024-03-11 00:00:00.0000000,38,SMSU,Secretaria Municipal de Segurança Urbana,10,...,1,500,9001,0,Leste,Subprefeitura Itaim Paulista,Supra-Distrital,Despesa Regionalizável,95.17,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1103229,01,9209,2025,224104,2025,08/29/2025 00:00:00.0000000,47,SUB-MG,Subprefeitura Vila Maria/Vila Guilherme,10,...,1,500,9001,0,Norte,Subprefeitura Vila Maria/Vila Guilherme,Supra-Distrital,Despesa Regionalizável,10223.84,2025
1103230,01,9209,2025,256339,2025,10/02/2025 00:00:00.0000000,47,SUB-MG,Subprefeitura Vila Maria/Vila Guilherme,10,...,1,500,9001,0,Norte,Subprefeitura Vila Maria/Vila Guilherme,Supra-Distrital,Despesa Regionalizável,10223.84,2025
1103231,01,9209,2025,272212,2025,10/23/2025 00:00:00.0000000,47,SUB-MG,Subprefeitura Vila Maria/Vila Guilherme,10,...,1,500,9001,0,Norte,Subprefeitura Vila Maria/Vila Guilherme,Supra-Distrital,Despesa Regionalizável,15143.04,2025
1103232,01,9209,2025,340085,2025,12/26/2025 00:00:00.0000000,47,SUB-MG,Subprefeitura Vila Maria/Vila Guilherme,10,...,1,500,9001,0,Norte,Subprefeitura Vila Maria/Vila Guilherme,Supra-Distrital,Despesa Regionalizável,15143.04,2025


Vamos conferir qual o percentual do orçamento está presente na tabela de detalhamento e regionalizado a nível de subprefeitura.

In [8]:
df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'].sum()/df_orcamento['Vl_Liquidado'].sum()

0.9584599475845688

95,8% de detalhamento é um percentual excelente. Vamos checar a regionalização.

In [9]:
(
    df_orcamento_r
    .assign(regionalizado=df_orcamento_r['SUBPREFEITURA'].str.contains('Supra')==False)
    .groupby('regionalizado')
    ['VALOR_DETALHAMENTO_AÇÃO'].sum()/df_orcamento['Vl_Liquidado'].sum()
)

regionalizado
False    0.150735
True     0.807725
Name: VALOR_DETALHAMENTO_AÇÃO, dtype: object

O percentual de regionalização é de 80,8%. Menor, mas ainda parece ser satisfatório. Vamos manter as informações de regionalização também.

# Exportando os arquivos

Neste notebook, vamos apenas salvar os arquivos extraídos na pasta de entrada de dados.

In [11]:
output_dir = path.join('data', 'cache', 'urbanismo')

if not path.exists(output_dir):
    makedirs(output_dir)

for c, df in [('orcamento_urbanismo_original', df_orcamento),
               ('orcamento_regionalizado_urbanismo_original', df_orcamento_r),
               ]:
    filename=path.join(output_dir, c)
    df.to_csv(f'{filename}.csv',
              sep=';',
              decimal=',',
              encoding='utf8',
              index=False
              )